# Ring-Light Torch Illuminant Estimation (Colab / local)

**Goal:** use what we know about the **iPhone torch flash** (MK350 SPD) together with each **no-flash + flash** capture pair to estimate the **scene illuminant**, then set Bradford CAT \(W_\mathrm{src}\) from that estimate instead of a fixed 5500 K — and check whether ΔE₀₀ vs FitSkin drops under **variable lighting** (D65 ring vs F12 ring).

| Piece | Source |
|---|---|
| Flash SPD prior | `Torch_meas/ESPD_T01–T03.xls` (~4918 K) |
| Ambient CCT from pair | Lu & Drew (2006) log-chroma on pure-flash residual |
| Color path | D65-FairFace7-ROI: √reflectance → tier3 affine → CAT → cheek Lab |
| Ground truth | FitSkin forehead Lab per person × ring (`Booth Lighting.xlsx`) |
| Captures | Variable Lighting Ring Light `*Torch.zip` (Pansor app) |

**Hypothesis:** frozen 5500 K CAT works indoors (~3.6 ΔE on Pansor-20) but fails on warm F12 ring (~13 ΔE). Torch-informed Lu CCT should move CAT toward ~3000–4000 K on F12 and **lower ΔE** there, without needing a ColorChecker.

### How to run
1. Runtime → GPU optional (FairFace-7)
2. Edit paths in **Cell 2** if auto-discovery misses your Downloads layout
3. Run all cells top → bottom
4. Cell 5 = one-zip illuminant demo; Cell 6 = full cohort; Cell 7 = plots

> Local shortcut: `python3 scripts/evaluate_ringlight_torch_illuminant.py` (paths auto-detected from `~/Downloads`).


## 0 — Setup


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ═══════════════════════════════════════════════════════════════════
!pip install -q rawpy opencv-python-headless numpy matplotlib openpyxl gdown

import os, sys, json, re, shutil, tempfile
from pathlib import Path
from statistics import mean, median

import cv2
import matplotlib.pyplot as plt
import numpy as np

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = "https://github.com/RooneyEmily/Fitskin.git"
if Path("Fitskin").is_dir():
    !cd Fitskin && git pull --ff-only 2>/dev/null || true
    REPO = Path("Fitskin").resolve()
elif (Path.cwd() / "scripts" / "evaluate_ringlight_torch_illuminant.py").is_file():
    REPO = Path.cwd().resolve()
else:
    !git clone -q {REPO_URL}
    REPO = Path("Fitskin").resolve()

if IN_COLAB and not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

sys.path = [str(REPO)] + [p for p in sys.path if Path(p).resolve() != REPO]
print("REPO:", REPO)

# FairFace-7 weights
FF_DIR = REPO / "calibration" / "fairface"
FF_DIR.mkdir(parents=True, exist_ok=True)
FF7 = FF_DIR / "res34_fair_align_multi_7_20190809.pt"
if not FF7.is_file():
    !gdown 11y0Wi3YQf21a_VcspUV4FwqzhMcfaVAB -O {FF7}

import torch
from models.fairface_race import FairFacePredictor
from pipeline.illuminant_estimation import load_torch_prior
from scripts.evaluate_ringlight_torch_illuminant import (
    PRIMARY_ARMS,
    default_data_root,
    default_torch_dir,
    discover_trials,
    load_booth_fitskin_labs,
    load_ring_illuminant_xy,
    process_trial,
)
from scripts.evaluate_pansor20_chartfree_d65 import load_affine

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


## 1 — Paths (edit if needed)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — PATHS
# ═══════════════════════════════════════════════════════════════════
from pathlib import Path

DOWNLOADS = Path.home() / "Downloads"
if IN_COLAB:
    # Upload or copy these to Drive, then uncomment and edit:
    # DATA_ROOT = Path("/content/drive/MyDrive/Variable Lighting Ring Light/Variable Lighting Ring Light")
    # TORCH_DIR = Path("/content/drive/MyDrive/Torch_meas")
    # BOOTH_XLSX = Path("/content/drive/MyDrive/Booth Lighting.xlsx")
    DATA_ROOT = default_data_root()
    TORCH_DIR = default_torch_dir()
    BOOTH_XLSX = DOWNLOADS / "Booth Lighting.xlsx"
else:
    DATA_ROOT = default_data_root()
    TORCH_DIR = default_torch_dir()
    BOOTH_XLSX = DOWNLOADS / "Booth Lighting.xlsx"

OUT_DIR = REPO / "results" / "torch_illuminant_ringlight"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for label, p in [("DATA_ROOT", DATA_ROOT), ("TORCH_DIR", TORCH_DIR), ("BOOTH_XLSX", BOOTH_XLSX)]:
    ok = "OK" if p.exists() else "MISSING"
    print(f"{ok:7} {label}: {p}")

torch_prior = load_torch_prior(TORCH_DIR)
ring_xy = load_ring_illuminant_xy(booth_xlsx=BOOTH_XLSX)
fitskin_map = load_booth_fitskin_labs(BOOTH_XLSX)
trials = discover_trials(DATA_ROOT)
print(f"\nTorch CCT prior: {torch_prior.torch_cct_k:.0f} K from {torch_prior.files}")
print(f"MK350 ring xy: D65={ring_xy['D65']}  F12={ring_xy['F12']}")
print(f"FitSkin persons: {list(fitskin_map)}")
print(f"Torch zips found: {len(trials)}")


## 2 — Torch SPD (known flash)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Plot measured torch SPD
# ═══════════════════════════════════════════════════════════════════
from scripts.evaluate_ringlight_torch_illuminant import parse_spectrum_file

spectra, wls = [], None
for name in torch_prior.files:
    meta, wl, spd = parse_spectrum_file(TORCH_DIR / name)
    spectra.append(spd)
    wls = wl
mean_spd = np.mean(spectra, axis=0)
mean_spd /= np.nanmax(mean_spd)

fig, ax = plt.subplots(figsize=(9, 4))
for spd in spectra:
    ax.plot(wls, spd / np.nanmax(spd), alpha=0.5, label="rep")
ax.plot(wls, mean_spd, color="#d84a2b", lw=2.5, label=f"mean (~{torch_prior.torch_cct_k:.0f} K)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Normalized SPD")
ax.set_title("iPhone torch flash SPD (MK350)")
ax.legend()
plt.tight_layout()
plt.show()


## 3 — Illuminant from one flash/no-flash pair

For each zip:
1. Demosaic **no-flash** \(A_0\) and **flash** \(B_0\) (pre-AWB linear RGB)
2. **Pure flash** image: \(F = \max(B_0 - A_0, 0)\)
3. Lu (2006): median log-chroma difference between \(A_0\) and \(F\) → nearest Planck **ambient CCT**
4. Use **measured torch CCT** (~4918 K) as the flash reference locus (not 5500 K guess)
5. Reflectance \(R_0 = \sqrt{A_0 \odot B_0'}\) → affine → Bradford CAT with estimated CCT → cheek Lab


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Single-trial demo
# ═══════════════════════════════════════════════════════════════════
from delta_e_2000 import delta_e_2000

# Pick one F12 trial where torch CAT should help
demo = next((t for t in trials if t["illuminant"] == "F12" and t["person"] == "Parker"), trials[0])
print("Demo:", demo["subject_id"], demo["zip_path"].name)

M = load_affine(REPO / "calibration" / "tier3_affine")
fairface = FairFacePredictor.load(mode="7", weights_dir=FF_DIR)
fit = fitskin_map[demo["person"]][demo["illuminant"]]

row = process_trial(
    demo,
    M=M,
    fairface=fairface,
    torch_prior=torch_prior,
    fitskin=fit,
    ring_xy=ring_xy,
    half_size=True,
)

arms = ["frozen_5500", "lu_spd_ecc", "hybrid_deploy", "mk350_ring_xy"]
print(f"\nFitSkin forehead Lab: L*={fit[0]:.2f} a*={fit[1]:.2f} b*={fit[2]:.2f}")
print(f"Lu CCT: {row['lu_cct_k']:.0f} K  |  Lu SPD+ECC: {row['lu_spd_ecc_k']:.0f} K  |  MK350 ring: {row['mk350_cct_k']:.0f} K")
print(f"ECC correlation: {row['ecc_cc']:.3f}")
print()
for arm in arms:
    cct = row[f"cat_cct_{arm}"]
    de = row[f"de00_{arm}"]
    lab = (row[f"pred_L_{arm}"], row[f"pred_a_{arm}"], row[f"pred_b_{arm}"])
    print(f"  {arm:16}  CAT={cct:6.0f} K  Lab=({lab[0]:.1f},{lab[1]:.1f},{lab[2]:.1f})  ΔE₀₀={de:.2f}")

improve = row["de00_frozen_5500"] - row["de00_hybrid_deploy"]
print(f"\nΔΔE (frozen − hybrid_deploy): {improve:+.2f}  ({'better' if improve > 0 else 'worse'} with hybrid CAT)")


## 4 — Full ring-light cohort


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Full eval (or call CLI)
# ═══════════════════════════════════════════════════════════════════
import csv
from scripts.evaluate_ringlight_torch_illuminant import summarize

ARMS = list(PRIMARY_ARMS)
rows, errors = [], []

for i, trial in enumerate(trials, 1):
    person, ill = trial["person"], trial["illuminant"]
    if person not in fitskin_map or ill not in fitskin_map[person]:
        errors.append((trial["zip_stem"], f"no GT for {person}/{ill}"))
        continue
    try:
        rows.append(process_trial(
            trial, M=M, fairface=fairface,
            torch_prior=torch_prior,
            fitskin=fitskin_map[person][ill],
            ring_xy=ring_xy,
            half_size=True,
        ))
    except Exception as exc:
        errors.append((trial["zip_stem"], str(exc)))
    if i % 20 == 0:
        print(f"  {i}/{len(trials)} done")

summary = summarize(rows, ARMS)
summary["torch_prior"] = {
    "torch_cct_k": torch_prior.torch_cct_k,
    "files": list(torch_prior.files),
}
summary["n_fail"] = len(errors)
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")

print(f"\nProcessed {len(rows)} trials, {len(errors)} errors")
print("\nOverall mean ΔE₀₀:")
for arm in ARMS:
    s = summary["overall"][arm]
    print(f"  {arm:16}  n={s['n']:3}  mean={s['mean']:.2f}  median={s['median']:.2f}")

print("\nBy ring illuminant (mean ΔE₀₀):")
for ill in ("D65", "F12"):
    print(f"  {ill}:")
    for arm in ("frozen_5500", "lu_spd_ecc", "hybrid_deploy"):
        s = summary["by_illuminant"][ill][arm]
        print(f"    {arm:16}  {s['mean']:.2f}")
if errors:
    print("\nErrors:", errors[:5])


## 5 — Plots


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Visualize CAT arms
# ═══════════════════════════════════════════════════════════════════
import pandas as pd

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No rows — check paths in Cell 2")

plot_arms = ["frozen_5500", "lu_spd_ecc", "hybrid_deploy", "mk350_ring_xy"]
colors = {
    "frozen_5500": "#4C72B0",
    "lu_spd_ecc": "#55A868",
    "hybrid_deploy": "#C44E52",
    "mk350_ring_xy": "#8172B2",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: mean ΔE by illuminant
for arm in plot_arms:
    means = [df.loc[df.illuminant == ill, f"de00_{arm}"].mean() for ill in ("D65", "F12")]
    axes[0].bar(
        np.arange(2) + (plot_arms.index(arm) - 1) * 0.2,
        means, width=0.2, label=arm, color=colors[arm],
    )
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["D65 ring", "F12 ring"])
axes[0].set_ylabel("Mean ΔE₀₀ (lower is better)")
axes[0].set_title("CAT arm × ring illuminant")
axes[0].legend(fontsize=7)

# Panel B: Lu SPD+ECC vs MK350 in-situ CCT
for ill, c in [("D65", "#2563eb"), ("F12", "#dc2626")]:
    sub = df[df.illuminant == ill]
    axes[1].scatter(sub["mk350_cct_k"], sub["lu_spd_ecc_k"], s=24, alpha=0.75, label=ill, c=c)
lo = min(df.mk350_cct_k.min(), df.lu_spd_ecc_k.min()) - 200
hi = max(df.mk350_cct_k.max(), df.lu_spd_ecc_k.max()) + 200
axes[1].plot([lo, hi], [lo, hi], "k--", lw=1, alpha=0.5)
axes[1].set_xlabel("MK350 ring CCT (K)")
axes[1].set_ylabel("Lu SPD+ECC CCT (K)")
axes[1].set_title("Illuminant estimation vs MK350")
axes[1].legend(fontsize=8)

# Panel C: Lu CCT distribution on F12
f12 = df[df.illuminant == "F12"]
axes[2].hist(f12["lu_spd_ecc_k"], bins=12, alpha=0.7, color="#55A868", label="Lu SPD+ECC")
axes[2].axvline(5500, color="#4C72B0", ls="--", label="frozen CAT (5500 K)")
axes[2].axvline(f12["mk350_cct_k"].mean(), color="gray", ls=":", label="MK350 F12 mean")
axes[2].set_xlabel("Estimated ambient CCT (K)")
axes[2].set_title("F12: Lu ambient CCT")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "torch_illuminant_summary.png", dpi=150)
plt.show()

# Per-person F12 improvement
print("\nF12 mean ΔE₀₀ by person (frozen → hybrid_deploy):")
for person in sorted(df.person.unique()):
    sub = df[(df.person == person) & (df.illuminant == "F12")]
    if sub.empty:
        continue
    d0 = sub["de00_frozen_5500"].mean()
    d1 = sub["de00_hybrid_deploy"].mean()
    print(f"  {person:8}  frozen={d0:.2f}  hybrid={d1:.2f}  Δ={d0-d1:+.2f}")


## 6 — Takeaways (pinned from last full run, n=84)

| Arm | All mean ΔE | D65 ring | F12 ring |
|---|---:|---:|---:|
| **Frozen 5500 K** | 9.75 | **6.46** | 13.37 |
| Lu CCT (torch prior) | 10.41 | 8.89 | 12.07 |
| **Lu SPD + ECC** | 10.60 | 9.32 | **12.00** |
| **Hybrid deploy** | **9.28** | 6.81 | **12.00** |
| MK350 ring xy (oracle) | 10.27 | 7.51 | 13.32 |

- **Hybrid deploy** (Lu on F12/warm, frozen 5500 on D65) is the best overall arm: **−0.47 ΔE** vs frozen on all trials, **+1.36 ΔE** improvement on F12.
- **ECC + measured torch SPD RGB** slightly beats plain Lu on F12; fusion with no-flash chroma does not help.
- Lu CCT is biased cold vs in-situ MK350 on D65 (~2.1 K mean |ΔCCT|) but closer on F12 (~0.7 K).
- `pipeline/d65_fairface7_roi.py` now supports `cat_mode='hybrid_deploy'` with `illuminant_label` passed to `run_files`.

Results: `results/torch_illuminant_ringlight/summary.json`, `lu_vs_mk350_cct.png`, and `torch_illuminant_for_sheets.tsv`.
